# DNA Storage Pipeline: Train DNN + Multi-Threshold CPL Analysis

Unified: Training, DNN Evaluation, CPL correction with multiple thresholds.
Change `DATASET_NAME` and `RUN_MODE` to control behavior.

**Key features:**
- Auto-detects `clustersizethreshold` using DNAformer's methodology
- Sweeps multiple CPL percentile thresholds (p0..p25)
- Generates paper-ready figures + .mat files for MATLAB
- Multi-run variance analysis for DNN inference


In [1]:
# =============================================================================
# IMPORTS
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
import os
import numpy as np
from tqdm.notebook import tqdm
import random
import time
import pandas as pd
import math
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib
import scipy.io as sio
matplotlib.rcParams.update({
    'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'serif',
})

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.1.2
Using device: cuda
GPU: NVIDIA GeForce RTX 3080


In [2]:
# =============================================================================
# CONFIGURATION
# =============================================================================

DATASET_NAME = "BinnedNanoporeTwoFlowcells"

# "train_and_cpl" | "train_only" | "cpl_only"
RUN_MODE = "cpl_only"

# CPL: Multi-threshold sweep
CPL_PERCENTILES = [0, 5, 10, 15, 20, 25]

# ─── clustersizethreshold ───────────────────────────────────────────
# DNAformer paper (Supplementary §1.2, §7):
#   Illumina:  clustersizethreshold = 3  (confidencethreshold = 0)
#   Nanopore:  clustersizethreshold = 4  (confidencethreshold = 0.7)
# Derived by scanning small cluster sizes upward and finding where
# DNN success rate drops below 50%.
#
# Set None → auto-detect (DNAformer methodology). Set int → override.
CLUSTERSIZE_THRESHOLD_OVERRIDE = None

# =============================================================================
# DATASET CONFIGURATIONS
# =============================================================================

DATASET_CONFIGS = {
    "Erlich": {
        "label_len": 152, "index_length": 16, "max_deviation": 10,
        "filter_index": False, "target_failure": 0.02,
        "description": "Erlich et al. (2017) - DNA Fountain, Illumina miSeq",
    },
    "Grass": {
        "label_len": 117, "index_length": 13, "max_deviation": 11,
        "filter_index": False, "target_failure": 0.66,
        "description": "Grass et al. (2015) - CustomArray, Illumina miSeq",
    },
    "Organick": {
        "label_len": 110, "index_length": 33, "max_deviation": 5,
        "filter_index": False, "target_failure": 0.17,
        "description": "Organick et al. (2018) - Twist, Illumina NextSeq",
    },
    "Srinivasavaradhan": {
        "label_len": 110, "index_length": 4, "max_deviation": 10,
        "filter_index": False, "target_failure": 14.58,
        "description": "Srinivasavaradhan et al. - Twist, MinION",
    },
    "BinnedPilotIllumina": {
        "label_len": 128, "index_length": 12, "max_deviation": 4,
        "filter_index": True, "target_failure": 0.01,
        "description": "DNAformer Pilot Illumina",
    },
    "BinnedPilotNanopore": {
        "label_len": 128, "index_length": 12, "max_deviation": 10,
        "filter_index": True, "target_failure": 1.0,
        "description": "DNAformer Pilot Nanopore",
    },
    "BinnedTestIllumina": {
        "label_len": 128, "index_length": 12, "max_deviation": 4,
        "filter_index": True, "target_failure": 0.0055,
        "description": "DNAformer Test Illumina",
    },
    "BinnedNanoporeFirstFlowcell": {
        "label_len": 128, "index_length": 12, "max_deviation": 4,
        "filter_index": True, "target_failure": 3.88,
        "description": "DNAformer Nanopore Single Flowcell",
    },
    "BinnedNanoporeSecondFlowcell": {
        "label_len": 128, "index_length": 12, "max_deviation": 4,
        "filter_index": True, "target_failure": 4.5,
        "description": "DNAformer Nanopore Second Flowcell",
    },
    "BinnedNanoporeTwoFlowcells": {
        "label_len": 128, "index_length": 12, "max_deviation": 4,
        "filter_index": True, "target_failure": 1.65,
        "description": "DNAformer Nanopore Two Flowcells Combined",
    },
}

DATASET_FILES = {
    "Erlich": ("binned_synthetic_erlich.txt", "Erlich.txt"),
    "Grass": ("binned_synthetic_grass.txt", "Grass.txt"),
    "Organick": ("binned_synthetic_organick.txt", None),
    "Srinivasavaradhan": ("binned_synthetic_srinivasavaradhan.txt", "Srinivasavaradhan.txt"),
    "BinnedPilotIllumina": ("binned_synthetic_dnaformer_illumina_pilot.txt", "BinnedPilotIllumina.txt"),
    "BinnedPilotNanopore": ("binned_synthetic_dnaformer_nanopore_pilot.txt", "BinnedPilotNanopore.txt"),
    "BinnedTestIllumina": ("binned_synthetic_dnaformer_illumina_full.txt", "BinnedTestIllumina.txt"),
    "BinnedNanoporeFirstFlowcell": ("binned_synthetic_dnaformer_nanopore_full.txt", "BinnedNanoporeFirstFlowcell.txt"),
    "BinnedNanoporeSecondFlowcell": ("binned_synthetic_dnaformer_nanopore_full.txt", "BinnedNanoporeSecondFlowcell.txt"),
    "BinnedNanoporeTwoFlowcells": ("binned_synthetic_dnaformer_nanopore_full.txt", "BinnedNanoporeTwoFlowcells.txt"),
}

# =============================================================================
# DERIVED PARAMETERS
# =============================================================================

CONFIG = DATASET_CONFIGS[DATASET_NAME]
LABEL_SEQ_LEN = CONFIG["label_len"]
INDEX_LENGTH = CONFIG["index_length"]
MAX_DEVIATION = CONFIG["max_deviation"]
FILTER_INDEX = CONFIG["filter_index"]
MAX_CLUSTER_SIZE = 16

if FILTER_INDEX:
    ORIGINAL_SEQ_LEN = LABEL_SEQ_LEN + INDEX_LENGTH
else:
    ORIGINAL_SEQ_LEN = LABEL_SEQ_LEN

MAX_READ_LEN = ORIGINAL_SEQ_LEN + MAX_DEVIATION + 8

BATCH_SIZE = 200 
LEARNING_RATE = 5e-4
MIN_LR = 5e-8
WEIGHT_DECAY = 1e-5
EPOCHS = 100
WARMUP_EPOCHS = 10
PATIENCE = 10

EMBED_DIM = 300
ALIGNMENT_FILTERS = 128
EMBEDDING_FILTERS = 500
GRU_HIDDEN = 300
GRU_LAYERS = 2
DROPOUT = 0.1

# =============================================================================
# PATHS
# =============================================================================

SYNTHETIC_DATA_DIR = Path("../generated_data_corrected")
REAL_DATA_DIR = Path("../Data")

_train_file, _eval_file = DATASET_FILES.get(DATASET_NAME, (None, None))
TRAIN_FILE = SYNTHETIC_DATA_DIR / _train_file if _train_file else None
EVAL_FILE = REAL_DATA_DIR / _eval_file if _eval_file else None

EXPERIMENT_DIR = Path(f"./Experiments/{DATASET_NAME}_v4")
WEIGHTS_DIR = EXPERIMENT_DIR / 'Models'
RESULTS_DIR = EXPERIMENT_DIR / 'Results'
FIGURES_DIR = EXPERIMENT_DIR / 'Figures'
MAT_DIR = EXPERIMENT_DIR / 'MatFiles'

if RUN_MODE == "cpl_only":
    MODEL_PATH = Path(f"./Experiments/{DATASET_NAME}_v4/Models/best_model_{DATASET_NAME}.pth")
    if not MODEL_PATH.exists():
        MODEL_PATH = WEIGHTS_DIR / f"best_model_{DATASET_NAME}.pth"
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"No model at {MODEL_PATH}")
    for d in [RESULTS_DIR, FIGURES_DIR, MAT_DIR]: d.mkdir(parents=True, exist_ok=True)
else:
    for d in [WEIGHTS_DIR, RESULTS_DIR, FIGURES_DIR, MAT_DIR]: d.mkdir(parents=True, exist_ok=True)
    MODEL_PATH = None

print(f"\n{'='*70}")
print(f"DATASET: {DATASET_NAME} | MODE: {RUN_MODE}")
print(f"{'='*70}")
print(f"  {CONFIG['description']}")
print(f"  LABEL_SEQ_LEN:   {LABEL_SEQ_LEN}")
print(f"  ORIGINAL_SEQ_LEN:{ORIGINAL_SEQ_LEN}")
print(f"  Target failure:  {CONFIG['target_failure']}%")
print(f"  CPL Percentiles: {CPL_PERCENTILES}")
print(f"  clustersizethreshold: {'auto-detect' if CLUSTERSIZE_THRESHOLD_OVERRIDE is None else CLUSTERSIZE_THRESHOLD_OVERRIDE}")
print(f"  Train: {TRAIN_FILE}")
print(f"  Eval:  {EVAL_FILE}")
print(f"{'='*70}\n")



DATASET: BinnedNanoporeTwoFlowcells | MODE: cpl_only
  DNAformer Nanopore Two Flowcells Combined
  LABEL_SEQ_LEN:   128
  ORIGINAL_SEQ_LEN:140
  Target failure:  1.65%
  CPL Percentiles: [0, 5, 10, 15, 20, 25]
  clustersizethreshold: auto-detect
  Train: ../generated_data_corrected/binned_synthetic_dnaformer_nanopore_full.txt
  Eval:  ../Data/BinnedNanoporeTwoFlowcells.txt



In [3]:
# =============================================================================
# VOCABULARY
# =============================================================================
 
VOCAB = {'N': 0, 'A': 1, 'C': 2, 'G': 3, 'T': 4}
PADDING_IDX = VOCAB['N']
VOCAB_SIZE = len(VOCAB)
INT_TO_CHAR = {i: c for c, i in VOCAB.items()}
DNA_BASES = ['A', 'C', 'G', 'T']
END_SYMBOL = 'S'
 
def encode_seq(seq, char_to_int, max_len, padding_idx):
    encoded = [char_to_int.get(c, padding_idx) for c in seq]
    encoded = encoded[:max_len]
    return encoded + [padding_idx] * (max_len - len(encoded))
 
def decode_seq(tensor, int_to_char):
    if tensor.is_cuda:
        tensor = tensor.cpu()
    ints = tensor.numpy().tolist()
    try:
        first_pad = ints.index(PADDING_IDX)
        ints = ints[:first_pad]
    except ValueError:
        pass
    return "".join([int_to_char.get(i, '?') for i in ints])


In [4]:
# =============================================================================
# DATASET
# =============================================================================
 
class DnaClusterDataset(Dataset):
    def __init__(self, filepath, max_cluster_size, max_read_len, label_seq_len,
                 char_to_int, padding_idx):
        self.max_cluster_size = max_cluster_size
        self.max_read_len = max_read_len
        self.label_seq_len = label_seq_len
        self.char_to_int = char_to_int
        self.padding_idx = padding_idx
        self.labels = []
        self.clusters = []
        self._load_data(filepath)
 
    def _load_data(self, filepath):
        print(f"Loading data from {filepath}...")
        with open(filepath, 'r') as f:
            content = f.read()
        blocks = content.split('\n\n')
        for block in tqdm(blocks, desc=f"Parsing {filepath.name}"):
            if not block.strip():
                continue
            lines = [l.strip() for l in block.strip().split('\n')]
            if len(lines) < 3:
                continue
            label_seq = lines[0]
            reads = [r for r in lines[2:] if r]
            if not reads or not label_seq:
                continue
            self.labels.append(label_seq)
            self.clusters.append(reads)
        print(f"Loaded {len(self.labels)} clusters.")
        # Verify label length
        if self.labels:
            actual = len(self.labels[0])
            print(f"  Label length in file: {actual}")
            if actual < self.label_seq_len:
                print(f"  ⚠️  WARNING: label in file ({actual}) < LABEL_SEQ_LEN ({self.label_seq_len})")
 
    def __len__(self):
        return len(self.labels)
 
    def __getitem__(self, idx):
        label_str = self.labels[idx]
        reads = list(self.clusters[idx])
        label_t = torch.tensor(
            encode_seq(label_str, self.char_to_int, self.label_seq_len, self.padding_idx),
            dtype=torch.long)
        cluster_t = torch.full((self.max_cluster_size, self.max_read_len),
                               self.padding_idx, dtype=torch.long)
        random.shuffle(reads)
        for i, r in enumerate(reads[:self.max_cluster_size]):
            cluster_t[i] = torch.tensor(
                encode_seq(r, self.char_to_int, self.max_read_len, self.padding_idx),
                dtype=torch.long)
        return cluster_t, label_t
 
    def get_raw_reads(self, idx): return self.clusters[idx]
    def get_raw_label(self, idx): return self.labels[idx]
    def get_cluster_size(self, idx): return len(self.clusters[idx])


In [5]:
# =============================================================================
# MODEL
# =============================================================================
 
class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding=0):
        super().__init__()
        self.depthwise = nn.Conv1d(in_channels, in_channels, kernel_size=kernel_size,
                                   padding=padding, groups=in_channels)
        self.pointwise = nn.Conv1d(in_channels, out_channels, kernel_size=1)
    def forward(self, x): return self.pointwise(self.depthwise(x))

class MultiKernelConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, seq_len, dropout=0.1):
        super().__init__()
        c1 = out_channels // 3
        c2 = out_channels // 3
        c3 = out_channels - c1 - c2
        self.conv1 = DepthwiseSeparableConv1d(in_channels, c1, kernel_size=1)
        self.conv3 = DepthwiseSeparableConv1d(in_channels, c2, kernel_size=3, padding=1)
        self.conv5 = DepthwiseSeparableConv1d(in_channels, c3, kernel_size=5, padding=2)
        self.norm1 = nn.LayerNorm([c1, seq_len])
        self.norm2 = nn.LayerNorm([c2, seq_len])
        self.norm3 = nn.LayerNorm([c3, seq_len])
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x1 = F.gelu(self.norm1(self.conv1(x)))
        x2 = F.gelu(self.norm2(self.conv3(x)))
        x3 = F.gelu(self.norm3(self.conv5(x)))
        return self.dropout(torch.cat([x1, x2, x3], dim=1))

class AlignmentModule(nn.Module):
    def __init__(self, embed_dim, out_channels, seq_len, dropout=0.1):
        super().__init__()
        self.conv_block1 = MultiKernelConvBlock(embed_dim, out_channels, seq_len, dropout)
        self.conv_block2 = MultiKernelConvBlock(out_channels, out_channels, seq_len, dropout)
    def forward(self, x):
        batch, cluster, emb, seq = x.shape
        x = x.view(batch * cluster, emb, seq)
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        return x.view(batch, cluster, -1, seq)

class EmbeddingModule(nn.Module):
    def __init__(self, in_channels, out_channels, in_len, out_len, dropout=0.1):
        super().__init__()
        self.conv_block = MultiKernelConvBlock(in_channels, out_channels, in_len, dropout)
        self.linear = nn.Linear(in_len, out_len)
    def forward(self, x):
        x = self.conv_block(x)
        batch, channels, seq_len = x.shape
        x = x.reshape(batch * channels, seq_len)
        x = self.linear(x)
        return x.reshape(batch, channels, -1)

class ImprovedDNAReconstructionModel(nn.Module):
    def __init__(self, vocab_size, label_seq_len, max_read_len, padding_idx,
                 embed_dim=128, alignment_filters=128, embedding_filters=256,
                 gru_hidden=256, gru_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.alignment = AlignmentModule(embed_dim, alignment_filters, max_read_len, dropout)
        self.embedding_module = EmbeddingModule(
            alignment_filters, embedding_filters, max_read_len, label_seq_len, dropout)
        self.gru = nn.GRU(embedding_filters, gru_hidden, num_layers=gru_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if gru_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(gru_hidden * 2, vocab_size)
 
    def forward(self, cluster_batch, return_probs=False):
        emb = self.embedding(cluster_batch).permute(0, 1, 3, 2)
        aligned = self.alignment(emb)
        nci = torch.sum(aligned, dim=1)
        feats = self.embedding_module(nci).permute(0, 2, 1)
        gru_out, _ = self.gru(feats)
        logits = self.fc_out(self.dropout(gru_out))
        if return_probs:
            return logits, F.softmax(logits, dim=2)
        return logits


In [6]:
# =============================================================================
# SCHEDULER
# =============================================================================
 
class CosineAnnealingWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, max_lr, min_lr):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.current_epoch = 0
    def step(self):
        if self.current_epoch < self.warmup_epochs:
            lr = self.max_lr * (self.current_epoch + 1) / self.warmup_epochs
        else:
            progress = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + (self.max_lr - self.min_lr) * 0.5 * (1 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        self.current_epoch += 1
        return lr
    def get_last_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


In [7]:
# =============================================================================
# TRAINING UTILITIES
# =============================================================================
 
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc="Training", leave=False)
    for clusters, labels in loop:
        clusters, labels = clusters.to(device), labels.to(device)
        logits = model(clusters)
        loss = criterion(logits.view(-1, VOCAB_SIZE), labels.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)
 
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    loop = tqdm(loader, desc="Validating", leave=False)
    with torch.no_grad():
        for clusters, labels in loop:
            clusters, labels = clusters.to(device), labels.to(device)
            logits = model(clusters)
            loss = criterion(logits.view(-1, VOCAB_SIZE), labels.view(-1))
            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)


In [8]:
# =============================================================================
# CPL ALGORITHM — Python translation of DNAformer C++
# =============================================================================
 
NINF = float('-inf')
NA = -1
 
def edit_distance_dp(X, Y):
    m, n = len(X), len(Y)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if X[i-1] == Y[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    return dp[m][n], dp
 
def edit_distance(X, Y): return edit_distance_dp(X, Y)[0]
def sum_ed(s, strs): return sum(edit_distance(s, t) for t in strs)
 
def backtrack_edit_vector(X, Y, dp):
    m, n = len(X), len(Y)
    ev = [""]*(2*m+1)
    mi, ni = m, n
    while True:
        if mi == 0: ev[0] = Y[:ni]; break
        elif ni == 0:
            for idx in range(mi): ev[2*idx+1] = ""
            break
        if X[mi-1] == Y[ni-1]: ev[2*mi-1] = X[mi-1]; mi -= 1; ni -= 1
        else:
            codes = []
            if dp[mi][ni] == dp[mi-1][ni-1]+1: codes.append(0)
            if dp[mi][ni] == dp[mi][ni-1]+1: codes.append(1)
            if dp[mi][ni] == dp[mi-1][ni]+1: codes.append(2)
            if len(codes) > 1: random.shuffle(codes)
            c = codes[0]
            if c == 0: ev[2*mi-1] = Y[ni-1]; mi -= 1; ni -= 1
            elif c == 1: ev[2*mi] = Y[ni-1] + ev[2*mi]; ni -= 1
            else: ev[2*mi-1] = ""; mi -= 1
    return ev
 
def edit_vector(X, Y):
    _, dp = edit_distance_dp(X, Y)
    return backtrack_edit_vector(X, Y, dp)
 
def anchor_w_all_pairs_ev(anchor_index, reads):
    anchor = reads[anchor_index]
    return [edit_vector(anchor, reads[i]) for i in range(len(reads)) if i != anchor_index]
 
def add_conditional_freq(ev, cfv):
    for i in range(len(ev)-1):
        fs, ts = ev[i], ev[i+1]
        if fs not in cfv[i]: cfv[i][fs] = ({}, 0)
        tc, tot = cfv[i][fs]; tc[ts] = tc.get(ts, 0)+1; cfv[i][fs] = (tc, tot+1)
    last = len(ev)-1; fs = ev[last]
    if fs not in cfv[last]: cfv[last][fs] = ({}, 0)
    tc, tot = cfv[last][fs]; tc[END_SYMBOL] = tc.get(END_SYMBOL, 0)+1; cfv[last][fs] = (tc, tot+1)

def conditional_freq_vec(all_ev):
    cf = [{} for _ in range(len(all_ev[0]))]
    for ev in all_ev: add_conditional_freq(ev, cf)
    return cf
 
def freq_to_prob_log(fv):
    pv = [{} for _ in range(len(fv))]
    tf = sum(v[1] for v in fv[0].values())
    for fs, (tc, fc) in fv[0].items():
        pv[0][fs] = {ts: math.log(c/tf) for ts, c in tc.items()}
    for i in range(1, len(fv)):
        for fs, (tc, fc) in fv[i].items():
            pv[i][fs] = {ts: math.log(c/fc) for ts, c in tc.items()}
    return pv
 
def prob_vec(anchor_index, reads):
    aev = anchor_w_all_pairs_ev(anchor_index, reads)
    if not aev: return None
    return freq_to_prob_log(conditional_freq_vec(aev))
 
def count_letters(prob): return sum(len(s) for lv in prob for s in lv)
 
def cond_prob_to_graph(prob):
    V = count_letters(prob)+2; letters = ['']*V; adj = [[] for _ in range(V)]
    llv = {}; cv = []; vi = V-1
    letters[vi] = END_SYMBOL; llv[END_SYMBOL] = vi; vi -= 1
    for level in reversed(prob):
        he = False; clv = {}
        for cs, cm in level.items():
            sl = len(cs)
            if sl == 0: he = True; continue
            elif sl == 1:
                letters[vi] = cs[0]
                for ts, wt in cm.items():
                    if len(ts) == 0:
                        for c, w in cv: adj[vi].append((c, w+wt))
                    elif ts in llv: adj[vi].append((llv[ts], wt))
                clv[cs] = vi; vi -= 1
            else:
                letters[vi] = cs[-1]
                for ts, wt in cm.items():
                    if len(ts) == 0:
                        for c, w in cv: adj[vi].append((c, w+wt))
                    elif ts in llv: adj[vi].append((llv[ts], wt))
                vi -= 1
                for si in range(sl-2, -1, -1):
                    letters[vi] = cs[si]; adj[vi].append((vi+1, 0.0)); vi -= 1
                clv[cs] = vi+1
        if he and "" in level:
            em = level[""]
            if "" not in em: cv = []
            else: ew = em[""]; cv = [(c, w+ew) for c, w in cv]
            for ts, wt in em.items():
                if ts == "": continue
                if ts in llv: cv.append((llv[ts], wt))
        else: cv = []
        llv = clv
    assert vi == 0; letters[0] = END_SYMBOL
    for s, v in llv.items(): adj[0].append((v, 0.0))
    for c, w in cv: adj[0].append((c, w))
    return letters, adj, V
 
def count_strings(prob): return sum(len(lv) for lv in prob)
 
def cond_prob_to_strings_graph(prob):
    V = count_strings(prob)+2; vts = [""]*V; adj = [[] for _ in range(V)]
    vts[0] = END_SYMBOL; vts[V-1] = END_SYMBOL
    s2v = [{} for _ in range(len(prob))]; vx = 1
    for li in range(len(prob)):
        for fs in prob[li]: vts[vx] = fs; s2v[li][fs] = vx; vx += 1
    for fs in prob[0]: adj[0].append((s2v[0][fs], 0.0))
    for li in range(len(prob)-1):
        for fs, tm in prob[li].items():
            fv = s2v[li][fs]
            for ts, wt in tm.items(): adj[fv].append((s2v[li+1][ts], wt))
    last = len(prob)-1
    for fs, tm in prob[last].items():
        fv = s2v[last][fs]
        for ts, wt in tm.items(): adj[fv].append((V-1, wt))
    return vts, adj, V
 
def min_max_letters_to_end(letters, adj, V):
    ev = V-1; ra = [[] for _ in range(V)]
    for u in range(V):
        for v, w in adj[u]: ra[v].append((u, 1))
    vis = [False]*V; to = []
    def dfs(v):
        stk = [(v, False)]
        while stk:
            nd, dn = stk.pop()
            if dn: to.append(nd); continue
            if vis[nd]: continue
            vis[nd] = True; stk.append((nd, True))
            for nb, w in ra[nd]:
                if not vis[nb]: stk.append((nb, False))
    for i in range(V):
        if not vis[i]: dfs(i)
    mnd = [float('inf')]*V; mnd[ev] = 0
    mxd = [float('-inf')]*V; mxd[ev] = 0
    for u in reversed(to):
        if mnd[u] != float('inf'):
            for nb, w in ra[u]:
                if mnd[nb] > mnd[u]+w: mnd[nb] = mnd[u]+w
        if mxd[u] != float('-inf'):
            for nb, w in ra[u]:
                if mxd[nb] < mxd[u]+w: mxd[nb] = mxd[u]+w
    mnl = [0]*V; mxl = [0]*V
    for v in range(V-1):
        mnl[v] = mnd[v]-1 if mnd[v] != float('inf') else 0
        mxl[v] = mxd[v]-1 if mxd[v] != float('-inf') else 0
    return mnl, mxl
 
def highest_score_fixed_len_path(letters, adj, mnl, mxl, tl):
    V = len(letters); ev = V-1; K = tl+1
    dp = [dict() for _ in range(K+1)]; dp[0][0] = (0.0, NA)
    for i in range(K):
        if not dp[i]: return ""
        for fv, (fval, _) in dp[i].items():
            for tv, wt in adj[fv]:
                lsf = i+1
                if tv != ev:
                    if lsf+mnl[tv] > tl: continue
                    if lsf+mxl[tv] < tl: continue
                nw = fval+wt
                if tv not in dp[i+1] or dp[i+1][tv][0] < nw: dp[i+1][tv] = (nw, fv)
    if ev not in dp[K]: return ""
    r = []; nv = ev
    for j in range(K, 1, -1): p = dp[j][nv][1]; r.append(letters[p]); nv = p
    r.reverse(); return "".join(r)
 
def longest_path_dag(adj, V, src, tgt):
    vis = [False]*V; to = []
    def dfs(v):
        stk = [(v, False)]
        while stk:
            nd, dn = stk.pop()
            if dn: to.append(nd); continue
            if vis[nd]: continue
            vis[nd] = True; stk.append((nd, True))
            for nb, w in adj[nd]:
                if not vis[nb]: stk.append((nb, False))
    for i in range(V):
        if not vis[i]: dfs(i)
    dist = [NINF]*V; par = [NA]*V; dist[src] = 0.0
    for u in reversed(to):
        if dist[u] == NINF: continue
        for nb, w in adj[u]:
            if dist[nb] < dist[u]+w: dist[nb] = dist[u]+w; par[nb] = u
    if dist[tgt] == NINF: return []
    path = []; c = tgt
    while c != NA: path.append(c); c = par[c] if c != src else NA
    if not path or path[-1] != src: path.append(src)
    path.reverse(); return path
 
def heaviest_path_string(vts, adj, V):
    path = longest_path_dag(adj, V, 0, V-1)
    if not path: return ""
    return "".join(vts[v] for v in path[1:-1])
 
def fixed_len_markov(reads, ai, tl):
    prob = prob_vec(ai, reads)
    if prob is None: return ""
    try:
        l, a, V = cond_prob_to_graph(prob)
        mn, mx = min_max_letters_to_end(l, a, V)
        return highest_score_fixed_len_path(l, a, mn, mx, tl)
    except: return ""

def heaviest_path(reads, ai):
    prob = prob_vec(ai, reads)
    if prob is None: return ""
    try:
        vts, a, V = cond_prob_to_strings_graph(prob)
        return heaviest_path_string(vts, a, V)
    except: return ""

def correct_cluster(reads, tl):
    corrected = []
    for i in range(len(reads)):
        g = fixed_len_markov(reads, i, tl)
        if not g: g = heaviest_path(reads, i)
        corrected.append(g)
    return corrected
 
def correct_cluster_twice(reads, tl):
    return correct_cluster(correct_cluster(reads, tl), tl)
 
def min_sum_ed_index(cands, orig):
    bi, bs = 0, float('inf')
    for i, c in enumerate(cands):
        s = sum_ed(c, orig)
        if s < bs: bs = s; bi = i
    return bi
 
def min_sum_ed_corrected_cluster_twice(reads, tl):
    if len(reads) == 1: return reads[0]
    corr = correct_cluster_twice(reads, tl)
    uniq = list(set(corr))
    if not uniq: return reads[0]
    return uniq[min_sum_ed_index(uniq, reads)]
 
def run_cpl_on_cluster(reads, original_length, output_length=None):
    """CPL: reconstruct full strand (original_length), output first output_length bases."""
    if output_length is None: output_length = original_length
    cleaned = [r.strip() for r in reads]
    valid = [r for r in cleaned if len(r) > 0 and set(r).issubset({'A','C','G','T'})]
    if len(valid) < 2:
        if valid: return valid[0][:output_length]
        return ''.join(random.choices(DNA_BASES, k=output_length))
    if len(valid) > MAX_CLUSTER_SIZE: valid = valid[:MAX_CLUSTER_SIZE]
    result = min_sum_ed_corrected_cluster_twice(valid, original_length)
    return result[:output_length]


In [9]:
# =============================================================================
# CONFIDENCE FILTER
# =============================================================================
 
class ConfidenceFilter:
    def __init__(self, threshold, min_cluster_for_cpl=4):
        self.threshold = threshold
        self.min_cluster = min_cluster_for_cpl
    def classify(self, probs, cluster_size):
        dna_probs = probs[:, 1:]
        m_M = dna_probs.max(dim=1).values.mean().item()
        if m_M >= self.threshold: return 'trust_dnn', m_M
        elif cluster_size <= self.min_cluster: return 'erasure', m_M
        else: return 'send_to_cpl', m_M


In [10]:
# =============================================================================
# TRAINING (skip if cpl_only)
# =============================================================================
 
if RUN_MODE in ("train_and_cpl", "train_only"):
    assert TRAIN_FILE and TRAIN_FILE.exists(), f"Training file not found: {TRAIN_FILE}"
 
    print(f"\n{'='*70}\nLOADING TRAINING DATA\n{'='*70}\n")
 
    full_train_dataset = DnaClusterDataset(
        TRAIN_FILE, MAX_CLUSTER_SIZE, MAX_READ_LEN,
        LABEL_SEQ_LEN, VOCAB, PADDING_IDX)
 
    val_size = int(0.02 * len(full_train_dataset))
    if val_size < BATCH_SIZE:
        val_size = min(BATCH_SIZE * 2, len(full_train_dataset) // 10)
    train_size = len(full_train_dataset) - val_size
    train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])
    print(f"Total: {len(full_train_dataset):,} | Train: {train_size:,} | Val: {val_size:,}")
 
    train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
 
    model = ImprovedDNAReconstructionModel(
        VOCAB_SIZE, LABEL_SEQ_LEN, MAX_READ_LEN, PADDING_IDX,
        EMBED_DIM, ALIGNMENT_FILTERS, EMBEDDING_FILTERS,
        GRU_HIDDEN, GRU_LAYERS, DROPOUT).to(DEVICE)
    print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
 
    criterion = nn.CrossEntropyLoss(ignore_index=PADDING_IDX, label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmupScheduler(optimizer, WARMUP_EPOCHS, EPOCHS, LEARNING_RATE, MIN_LR)
 
    print(f"\n{'='*70}\nTRAINING: {DATASET_NAME}\n{'='*70}\n")
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses, learning_rates = [], [], []
 
    for epoch in range(EPOCHS):
        t0 = time.time()
        current_lr = scheduler.get_last_lr()[0]
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss = validate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        train_losses.append(train_loss); val_losses.append(val_loss); learning_rates.append(current_lr)
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | LR: {current_lr:.2e} | "
              f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"Time: {time.time()-t0:.1f}s", end="")
        if val_loss < best_val_loss:
            imp = best_val_loss - val_loss; best_val_loss = val_loss
            torch.save(model.state_dict(), WEIGHTS_DIR / f"best_model_{DATASET_NAME}.pth")
            print(f" ✓ BEST (↓{imp:.4f})"); patience_counter = 0
        else: print(); patience_counter += 1
        if patience_counter >= PATIENCE: print(f"\nEarly stopping at epoch {epoch+1}"); break
 
    torch.save(model.state_dict(), WEIGHTS_DIR / f"final_model_{DATASET_NAME}.pth")
    MODEL_PATH = WEIGHTS_DIR / f"best_model_{DATASET_NAME}.pth"
    pd.DataFrame({'epoch': range(1, len(train_losses)+1), 'train_loss': train_losses,
                  'val_loss': val_losses, 'learning_rate': learning_rates
                  }).to_csv(RESULTS_DIR / f"history_{DATASET_NAME}.csv", index=False)
    print(f"\nBest val loss: {best_val_loss:.4f} | Models: {WEIGHTS_DIR}")
    if RUN_MODE == "train_only":
        print(f"\n{'='*70}\nTRAINING COMPLETE (train_only)\n{'='*70}")
else:
    print(f"Skipping training (RUN_MODE='{RUN_MODE}')")


Skipping training (RUN_MODE='cpl_only')


In [11]:
# =============================================================================
# LOAD MODEL + TEST DATA
# =============================================================================
 
if RUN_MODE != "train_only":
    assert EVAL_FILE and EVAL_FILE.exists(), f"Eval not found: {EVAL_FILE}"
    assert MODEL_PATH and os.path.isfile(str(MODEL_PATH)), f"Model not found: {MODEL_PATH}"
 
    print(f"\n{'='*70}\nLOADING MODEL\n{'='*70}")
    model = ImprovedDNAReconstructionModel(
        VOCAB_SIZE, LABEL_SEQ_LEN, MAX_READ_LEN, PADDING_IDX,
        EMBED_DIM, ALIGNMENT_FILTERS, EMBEDDING_FILTERS,
        GRU_HIDDEN, GRU_LAYERS, DROPOUT).to(DEVICE)
    model.load_state_dict(torch.load(str(MODEL_PATH), map_location=DEVICE, weights_only=True))
    model.eval()
    print(f"✓ Loaded ({sum(p.numel() for p in model.parameters()):,} params) from {MODEL_PATH}")
 
    test_ds = DnaClusterDataset(EVAL_FILE, MAX_CLUSTER_SIZE, MAX_READ_LEN,
                                LABEL_SEQ_LEN, VOCAB, PADDING_IDX)
    test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)



LOADING MODEL


/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


✓ Loaded (3,447,325 params) from Experiments/BinnedNanoporeTwoFlowcells_v4/Models/best_model_BinnedNanoporeTwoFlowcells.pth
Loading data from ../Data/BinnedNanoporeTwoFlowcells.txt...


Parsing BinnedNanoporeTwoFlowcells.txt:   0%|          | 0/109977 [00:00<?, ?it/s]

Loaded 109976 clusters.
  Label length in file: 140


In [12]:
# =============================================================================
# STAGE 0: MULTI-RUN VARIANCE ANALYSIS (DNN-only)
# =============================================================================
# DnaClusterDataset shuffles read order within each cluster at load time.
# This causes slight variation in DNN predictions across runs.
# We quantify this variance here, then proceed with the main single-run pipeline.
#
# Note: This cell is independent — it does NOT set variables used by later cells.

N_RUNS = 10  # Number of evaluation runs for variance estimation

if RUN_MODE != "train_only":
    _var_model_path = MODEL_PATH
    if _var_model_path is None:
        _var_model_path = WEIGHTS_DIR / f'best_model_{DATASET_NAME}.pth'
    assert _var_model_path and os.path.isfile(str(_var_model_path)), f'Model not found: {_var_model_path}'

    print(f"\n{'='*70}")
    print(f"STAGE 0: Multi-Run Variance Analysis ({N_RUNS} runs)")
    print(f"{'='*70}\n")

    _var_model = ImprovedDNAReconstructionModel(
        VOCAB_SIZE, LABEL_SEQ_LEN, MAX_READ_LEN, PADDING_IDX,
        EMBED_DIM, ALIGNMENT_FILTERS, EMBEDDING_FILTERS,
        GRU_HIDDEN, GRU_LAYERS, DROPOUT).to(DEVICE)
    _var_model.load_state_dict(torch.load(str(_var_model_path), map_location=DEVICE, weights_only=True))
    _var_model.eval()

    run_failures = []
    run_failure_indices = []

    for run in range(N_RUNS):
        random.seed(run)
        np.random.seed(run)

        _var_ds = DnaClusterDataset(EVAL_FILE, MAX_CLUSTER_SIZE, MAX_READ_LEN,
                                     LABEL_SEQ_LEN, VOCAB, PADDING_IDX)
        _var_loader = DataLoader(_var_ds, BATCH_SIZE, shuffle=False,
                                 num_workers=0, pin_memory=True)

        _preds, _labels = [], []
        with torch.no_grad():
            for clusters, labels in _var_loader:
                logits = _var_model(clusters.to(DEVICE))
                preds = logits.argmax(dim=2).cpu()
                for i in range(labels.shape[0]):
                    _preds.append(preds[i])
                    _labels.append(labels[i])

        N_var = len(_preds)
        failed_idx = [i for i in range(N_var)
                      if decode_seq(_preds[i], INT_TO_CHAR) != decode_seq(_labels[i], INT_TO_CHAR)]
        run_failures.append(len(failed_idx))
        run_failure_indices.append(set(failed_idx))
        print(f'  Run {run+1:>2}/{N_RUNS}: {len(failed_idx):>4} failures ({len(failed_idx)/N_var*100:.4f}%)')

    failures_arr = np.array(run_failures)
    rates_arr = failures_arr / N_var * 100

    # Identify stable vs flipping clusters
    all_ever_failed = set()
    for s in run_failure_indices:
        all_ever_failed |= s
    always_failed = set.intersection(*run_failure_indices) if run_failure_indices else set()
    sometimes_failed = all_ever_failed - always_failed

    print(f'\n  Summary over {N_RUNS} runs (N={N_var:,}):')
    print(f'    Failures:  {failures_arr.mean():.1f} ± {failures_arr.std():.1f}')
    print(f'    Rate:      {rates_arr.mean():.4f}% ± {rates_arr.std():.4f}%')
    print(f'    Range:     [{failures_arr.min()}, {failures_arr.max()}]')
    print(f'\n    Always fail (every run):     {len(always_failed)}')
    print(f'    Sometimes fail (flipping):   {len(sometimes_failed)}')
    print(f'    Total unique failures seen:  {len(all_ever_failed)}')

    # Save results
    pd.DataFrame({
        'run': range(1, N_RUNS+1),
        'failures': run_failures,
        'rate_pct': rates_arr.tolist(),
    }).to_csv(RESULTS_DIR / f'multirun_variance_{DATASET_NAME}.csv', index=False)

    sio.savemat(str(MAT_DIR / f'multirun_variance_{DATASET_NAME}.mat'), {
        'dataset_name': DATASET_NAME,
        'n_runs': np.float64(N_RUNS),
        'N': np.float64(N_var),
        'failures_per_run': failures_arr.astype(np.float64),
        'rates_per_run': rates_arr,
        'mean_failures': np.float64(failures_arr.mean()),
        'std_failures': np.float64(failures_arr.std()),
        'mean_rate_pct': np.float64(rates_arr.mean()),
        'std_rate_pct': np.float64(rates_arr.std()),
        'always_fail_count': np.float64(len(always_failed)),
        'sometimes_fail_count': np.float64(len(sometimes_failed)),
    })
    print(f'\n  → Saved to {RESULTS_DIR} and {MAT_DIR}')

    # Reset random state so subsequent cells are not affected
    random.seed()
    np.random.seed()

    # Clean up memory
    del _var_model, _var_ds, _var_loader, _preds, _labels
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    print(f'\n  For paper: "{rates_arr.mean():.4f}% ± {rates_arr.std():.4f}% (mean ± std, n={N_RUNS})"')



STAGE 0: Multi-Run Variance Analysis (10 runs)

Loading data from ../Data/BinnedNanoporeTwoFlowcells.txt...


Parsing BinnedNanoporeTwoFlowcells.txt:   0%|          | 0/109977 [00:00<?, ?it/s]

Loaded 109976 clusters.
  Label length in file: 140
  Run  1/10: 3114 failures (2.8315%)
Loading data from ../Data/BinnedNanoporeTwoFlowcells.txt...


Parsing BinnedNanoporeTwoFlowcells.txt:   0%|          | 0/109977 [00:00<?, ?it/s]

Loaded 109976 clusters.
  Label length in file: 140
  Run  2/10: 3142 failures (2.8570%)
Loading data from ../Data/BinnedNanoporeTwoFlowcells.txt...


...

Loaded 109976 clusters.
  Label length in file: 140
  Run  8/10: 3154 failures (2.8679%)
Loading data from ../Data/BinnedNanoporeTwoFlowcells.txt...


Parsing BinnedNanoporeTwoFlowcells.txt:   0%|          | 0/109977 [00:00<?, ?it/s]

Loaded 109976 clusters.
  Label length in file: 140
  Run  9/10: 3163 failures (2.8761%)
Loading data from ../Data/BinnedNanoporeTwoFlowcells.txt...


Parsing BinnedNanoporeTwoFlowcells.txt:   0%|          | 0/109977 [00:00<?, ?it/s]

Loaded 109976 clusters.
  Label length in file: 140
  Run 10/10: 3154 failures (2.8679%)

  Summary over 10 runs (N=109,976):
    Failures:  3136.5 ± 25.1
    Rate:      2.8520% ± 0.0228%
    Range:     [3101, 3181]

    Always fail (every run):     1833
    Sometimes fail (flipping):   3957
    Total unique failures seen:  5790

  → Saved to Experiments/BinnedNanoporeTwoFlowcells_v4/Results and Experiments/BinnedNanoporeTwoFlowcells_v4/MatFiles

  For paper: "2.8520% ± 0.0228% (mean ± std, n=10)"


In [13]:
# =============================================================================
# STAGE 1: DNN INFERENCE
# =============================================================================
 
if RUN_MODE != "train_only":
    print(f"\n{'='*70}\nSTAGE 1: DNN Inference\n{'='*70}\n")
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for clusters, labels in tqdm(test_loader, desc="DNN"):
            logits, probs = model(clusters.to(DEVICE), return_probs=True)
            preds = logits.argmax(dim=2).cpu(); probs_cpu = probs.cpu()
            for i in range(labels.shape[0]):
                all_preds.append(preds[i]); all_probs.append(probs_cpu[i]); all_labels.append(labels[i])
    N = len(all_preds)
    dnn_failed = sum(1 for i in range(N)
        if decode_seq(all_preds[i], INT_TO_CHAR) != decode_seq(all_labels[i], INT_TO_CHAR))
    dnn_rate = dnn_failed / N * 100
    print(f"  Total: {N:,}\n  DNN-only failures: {dnn_failed:,} ({dnn_rate:.4f}%)")



STAGE 1: DNN Inference



DNN:   0%|          | 0/550 [00:00<?, ?it/s]

  Total: 109,976
  DNN-only failures: 3,139 (2.8543%)


In [14]:
# =============================================================================
# PRE-COMPUTE (after DNN inference, before confidence analysis)
# =============================================================================

if RUN_MODE != "train_only":
    dnn_decoded = [decode_seq(all_preds[i], INT_TO_CHAR) for i in range(N)]
    label_decoded = [decode_seq(all_labels[i], INT_TO_CHAR) for i in range(N)]
    dnn_correct_mask = np.array([dnn_decoded[i] == label_decoded[i] for i in range(N)])
    dnn_fail_mask = ~dnn_correct_mask
    cluster_sizes = np.array([test_ds.get_cluster_size(i) for i in range(N)])


In [15]:
# =============================================================================
# STAGE 2a: Auto-detect clustersizethreshold
# =============================================================================
# DNAformer methodology (Supplementary §1.2, §7):
#   Scan cluster sizes from SMALL to LARGE.
#   Find the largest small cluster size where DNN success rate < 50%.
#   Illumina → 3, Nanopore → 4.
#
# Key: only consider small sizes with enough samples (≥20 clusters).
# A single failed cluster at size 630 should NOT set the threshold to 630.

if RUN_MODE != "train_only":
    print(f"\n{'='*70}")
    print(f"STAGE 2a: Cluster Size Analysis (auto-detect clustersizethreshold)")
    print(f"{'='*70}\n")

    MIN_SAMPLES_FOR_THRESHOLD = 20  # Need enough samples for reliable estimate

    size_correct = defaultdict(int)
    size_total = defaultdict(int)
    for i in range(N):
        cs = cluster_sizes[i]
        size_total[cs] += 1
        if dnn_correct_mask[i]:
            size_correct[cs] += 1

    all_sizes_sorted = sorted(size_total.keys())
    size_success_rate = {}
    print(f"  {'Size':>6} {'Total':>8} {'Correct':>8} {'Failed':>8} {'Rate':>10}")
    print(f"  {'-'*6} {'-'*8} {'-'*8} {'-'*8} {'-'*10}")
    for cs in all_sizes_sorted:
        tot = size_total[cs]
        cor = size_correct[cs]
        rate = cor / tot * 100 if tot > 0 else 0
        size_success_rate[cs] = rate
        marker = " ← <50%" if rate < 50 and tot >= MIN_SAMPLES_FOR_THRESHOLD else ""
        print(f"  {cs:>6} {tot:>8,} {cor:>8,} {tot-cor:>8,} {rate:>9.1f}%{marker}")

    if CLUSTERSIZE_THRESHOLD_OVERRIDE is not None:
        CLUSTERSIZE_THRESHOLD = CLUSTERSIZE_THRESHOLD_OVERRIDE
        print(f"\n  clustersizethreshold = {CLUSTERSIZE_THRESHOLD} (user override)")
    else:
        # DNAformer method: scan from smallest upward, stop when success >= 50%
        # Only consider sizes with enough samples
        CLUSTERSIZE_THRESHOLD = 0  # default: no erasures
        for cs in all_sizes_sorted:
            if size_total[cs] >= MIN_SAMPLES_FOR_THRESHOLD and size_success_rate[cs] < 50.0:
                CLUSTERSIZE_THRESHOLD = cs
            elif size_total[cs] >= MIN_SAMPLES_FOR_THRESHOLD and size_success_rate[cs] >= 50.0:
                break  # Stop at first reliable size with good performance
        print(f"\n  Auto-detected clustersizethreshold = {CLUSTERSIZE_THRESHOLD}")
        if CLUSTERSIZE_THRESHOLD == 0:
            print(f"  (No small cluster size has <50% success → DNN works well at all sizes)")

    # Save
    cs_df = pd.DataFrame([{
        'cluster_size': cs, 'total': size_total[cs], 'correct': size_correct[cs],
        'failed': size_total[cs] - size_correct[cs], 'success_rate_pct': size_success_rate[cs]
    } for cs in all_sizes_sorted])
    cs_df.to_csv(RESULTS_DIR / f'cluster_size_analysis_{DATASET_NAME}.csv', index=False)

    sio.savemat(str(MAT_DIR / f'figS1_clustersize_{DATASET_NAME}.mat'), {
        'dataset_name': DATASET_NAME,
        'cluster_sizes': np.array(all_sizes_sorted, dtype=np.float64),
        'success_rates': np.array([size_success_rate[cs] for cs in all_sizes_sorted], dtype=np.float64),
        'cluster_counts': np.array([size_total[cs] for cs in all_sizes_sorted], dtype=np.float64),
        'clustersize_threshold': np.float64(CLUSTERSIZE_THRESHOLD),
    })
    print(f"  → Saved CSV + .mat")



STAGE 2a: Cluster Size Analysis (auto-detect clustersizethreshold)

    Size    Total  Correct   Failed       Rate
  ------ -------- -------- -------- ----------
       1       17        0       17       0.0%
       2       26        3       23      11.5% ← <50%
       3       52       27       25      51.9%
       4      120       78       42      65.0%
       5      226      173       53      76.5%
       6      364      304       60      83.5%
       7      647      576       71      89.0%
       8      908      844       64      93.0%
       9    1,342    1,246       96      92.8%
      10    1,862    1,768       94      95.0%
      11    2,370    2,283       87      96.3%
      12    2,993    2,871      122      95.9%
      13    3,591    3,488      103      97.1%
      14    4,311    4,196      115      97.3%
      15    4,793    4,668      125      97.4%
      16    5,193    5,097       96      98.2%
      17    5,759    5,631      128      97.8%
      18    6,042    5,921     

In [16]:
# =============================================================================
# STAGE 2b: Confidence Analysis
# =============================================================================

if RUN_MODE != "train_only":
    print(f"\n{'='*70}")
    print(f"STAGE 2b: Confidence Analysis (clustersizethreshold={CLUSTERSIZE_THRESHOLD})")
    print(f"{'='*70}\n")

    m_scores = [all_probs[i][:, 1:].max(dim=1).values.mean().item() for i in range(N)]
    m_arr = np.array(m_scores)
    correct_scores = m_arr[dnn_correct_mask]
    fail_scores = m_arr[dnn_fail_mask] if dnn_failed > 0 else np.array([])

    print(f"  m(M): mean={m_arr.mean():.6f} median={np.median(m_arr):.6f} std={m_arr.std():.6f}")
    if dnn_failed > 0:
        print(f"  Correct: mean={correct_scores.mean():.6f}  Failed: mean={fail_scores.mean():.6f}")
        for idx, s in enumerate(fail_scores):
            pctile = np.searchsorted(np.sort(m_arr), s) / N * 100
            print(f"    Failure {idx+1}: m(M)={s:.6f}, percentile={pctile:.1f}%")
    else:
        print(f"  DNN has zero failures.")

    # Sweep preview
    print(f"\n  {'Pctl':>5} {'Thr':>10} {'Sent':>8} {'Era':>6} {'Capt':>6} {'AtRisk':>8} {'~Net':>6}")
    for pct in CPL_PERCENTILES:
        if pct == 0:
            print(f"  {'p0':>5} {'DNN-only':>10} {'0':>8} {'0':>6} {'0':>6} {'0':>8} {'0':>6}")
            continue
        thr = np.percentile(m_arr, pct)
        below = m_arr < thr
        can_cpl = below & (cluster_sizes > CLUSTERSIZE_THRESHOLD)
        era = below & (cluster_sizes <= CLUSTERSIZE_THRESHOLD)
        capt = int((can_cpl & dnn_fail_mask).sum()) if dnn_failed > 0 else 0
        at_risk = int((can_cpl & dnn_correct_mask).sum())
        est_net = capt - int(at_risk * 0.04)
        sign = '+' if est_net > 0 else ''
        print(f"  p{pct:>4} {thr:>10.4f} {int(can_cpl.sum()):>8,} {int(era.sum()):>6} {capt:>6} {at_risk:>8,} {sign}{est_net:>5}")

    sio.savemat(str(MAT_DIR / f'fig4_confidence_{DATASET_NAME}.mat'), {
        'dataset_name': DATASET_NAME,
        'm_scores_all': m_arr,
        'm_scores_correct': correct_scores,
        'm_scores_failed': fail_scores,
        'dnn_failed': np.float64(dnn_failed),
        'N': np.float64(N),
        'clustersize_threshold': np.float64(CLUSTERSIZE_THRESHOLD),
    })



STAGE 2b: Confidence Analysis (clustersizethreshold=2)

  m(M): mean=0.918972 median=0.920744 std=0.017668
  Correct: mean=0.920542  Failed: mean=0.865553
    Failure 1: m(M)=0.884143, percentile=1.1%
    Failure 2: m(M)=0.862107, percentile=0.9%
    Failure 3: m(M)=0.796699, percentile=0.5%
    Failure 4: m(M)=0.920376, percentile=12.3%
    Failure 5: m(M)=0.893715, percentile=1.2%
    Failure 6: m(M)=0.917491, percentile=2.8%
    Failure 7: m(M)=0.859186, percentile=0.9%
    Failure 8: m(M)=0.869877, percentile=1.0%
    Failure 9: m(M)=0.913604, percentile=2.0%
    Failure 10: m(M)=0.803149, percentile=0.5%
    Failure 11: m(M)=0.917875, percentile=3.0%
    Failure 12: m(M)=0.888104, percentile=1.2%
    Failure 13: m(M)=0.907408, percentile=1.6%
    Failure 14: m(M)=0.920296, percentile=9.5%
    Failure 15: m(M)=0.918857, percentile=3.7%
    Failure 16: m(M)=0.920922, percentile=75.4%
    Failure 17: m(M)=0.918795, percentile=3.7%
    Failure 18: m(M)=0.917141, percentile=2.7%
    F

    Failure 265: m(M)=0.821794, percentile=0.6%
    Failure 266: m(M)=0.920583, percentile=28.1%
    Failure 267: m(M)=0.921314, percentile=97.5%
    Failure 268: m(M)=0.922001, percentile=99.5%
    Failure 269: m(M)=0.912199, percentile=1.9%
    Failure 270: m(M)=0.664999, percentile=0.2%
    Failure 271: m(M)=0.816305, percentile=0.6%
    Failure 272: m(M)=0.918426, percentile=3.4%
    Failure 273: m(M)=0.918969, percentile=3.9%
    Failure 274: m(M)=0.920883, percentile=70.3%
    Failure 275: m(M)=0.916793, percentile=2.5%
    Failure 276: m(M)=0.915154, percentile=2.2%
    Failure 277: m(M)=0.914066, percentile=2.0%
    Failure 278: m(M)=0.921154, percentile=93.7%
    Failure 279: m(M)=0.870412, percentile=1.0%
    Failure 280: m(M)=0.858803, percentile=0.9%
    Failure 281: m(M)=0.916544, percentile=2.5%
    Failure 282: m(M)=0.920835, percentile=63.7%
    Failure 283: m(M)=0.653601, percentile=0.1%
    Failure 284: m(M)=0.905751, percentile=1.5%
    Failure 285: m(M)=0.907098, pe

    Failure 527: m(M)=0.918961, percentile=3.8%
    Failure 528: m(M)=0.921007, percentile=84.4%
    Failure 529: m(M)=0.697446, percentile=0.2%
    Failure 530: m(M)=0.916392, percentile=2.4%
    Failure 531: m(M)=0.865345, percentile=0.9%
    Failure 532: m(M)=0.861823, percentile=0.9%
    Failure 533: m(M)=0.881809, percentile=1.1%
    Failure 534: m(M)=0.915428, percentile=2.2%
    Failure 535: m(M)=0.889956, percentile=1.2%
    Failure 536: m(M)=0.920799, percentile=58.5%
    Failure 537: m(M)=0.802313, percentile=0.5%
    Failure 538: m(M)=0.918094, percentile=3.2%
    Failure 539: m(M)=0.917871, percentile=3.0%
    Failure 540: m(M)=0.919121, percentile=4.0%
    Failure 541: m(M)=0.594956, percentile=0.0%
    Failure 542: m(M)=0.921005, percentile=84.2%
    Failure 543: m(M)=0.920498, percentile=19.7%
    Failure 544: m(M)=0.918493, percentile=3.4%
    Failure 545: m(M)=0.916863, percentile=2.6%
    Failure 546: m(M)=0.768900, percentile=0.4%
    Failure 547: m(M)=0.917640, perc

    Failure 784: m(M)=0.857042, percentile=0.8%
    Failure 785: m(M)=0.858776, percentile=0.9%
    Failure 786: m(M)=0.900055, percentile=1.4%
    Failure 787: m(M)=0.920213, percentile=7.7%
    Failure 788: m(M)=0.914925, percentile=2.2%
    Failure 789: m(M)=0.918917, percentile=3.8%
    Failure 790: m(M)=0.652196, percentile=0.1%
    Failure 791: m(M)=0.919248, percentile=4.2%
    Failure 792: m(M)=0.847839, percentile=0.8%
    Failure 793: m(M)=0.919739, percentile=4.9%
    Failure 794: m(M)=0.920110, percentile=6.5%
    Failure 795: m(M)=0.904255, percentile=1.5%
    Failure 796: m(M)=0.921253, percentile=96.5%
    Failure 797: m(M)=0.910298, percentile=1.7%
    Failure 798: m(M)=0.909175, percentile=1.7%
    Failure 799: m(M)=0.918421, percentile=3.4%
    Failure 800: m(M)=0.919191, percentile=4.1%
    Failure 801: m(M)=0.919215, percentile=4.1%
    Failure 802: m(M)=0.920824, percentile=62.2%
    Failure 803: m(M)=0.919203, percentile=4.1%
    Failure 804: m(M)=0.914026, percen

    Failure 1043: m(M)=0.906389, percentile=1.5%
    Failure 1044: m(M)=0.919710, percentile=4.8%
    Failure 1045: m(M)=0.919852, percentile=5.2%
    Failure 1046: m(M)=0.822036, percentile=0.6%
    Failure 1047: m(M)=0.920207, percentile=7.6%
    Failure 1048: m(M)=0.835660, percentile=0.7%
    Failure 1049: m(M)=0.917576, percentile=2.9%
    Failure 1050: m(M)=0.920187, percentile=7.4%
    Failure 1051: m(M)=0.917540, percentile=2.9%
    Failure 1052: m(M)=0.914281, percentile=2.1%
    Failure 1053: m(M)=0.920125, percentile=6.7%
    Failure 1054: m(M)=0.770251, percentile=0.4%
    Failure 1055: m(M)=0.917779, percentile=3.0%
    Failure 1056: m(M)=0.920560, percentile=25.5%
    Failure 1057: m(M)=0.920435, percentile=15.3%
    Failure 1058: m(M)=0.871594, percentile=1.0%
    Failure 1059: m(M)=0.848791, percentile=0.8%
    Failure 1060: m(M)=0.917401, percentile=2.8%
    Failure 1061: m(M)=0.802301, percentile=0.5%
    Failure 1062: m(M)=0.914444, percentile=2.1%
    Failure 1063: 

...

    Failure 2078: m(M)=0.734997, percentile=0.3%
    Failure 2079: m(M)=0.916579, percentile=2.5%
    Failure 2080: m(M)=0.662507, percentile=0.2%
    Failure 2081: m(M)=0.917486, percentile=2.8%
    Failure 2082: m(M)=0.761322, percentile=0.4%
    Failure 2083: m(M)=0.904233, percentile=1.5%
    Failure 2084: m(M)=0.896750, percentile=1.3%
    Failure 2085: m(M)=0.920222, percentile=7.9%
    Failure 2086: m(M)=0.919092, percentile=4.0%
    Failure 2087: m(M)=0.856310, percentile=0.8%
    Failure 2088: m(M)=0.908362, percentile=1.6%
    Failure 2089: m(M)=0.917535, percentile=2.9%
    Failure 2090: m(M)=0.920491, percentile=19.1%
    Failure 2091: m(M)=0.919977, percentile=5.7%
    Failure 2092: m(M)=0.606084, percentile=0.1%
    Failure 2093: m(M)=0.916786, percentile=2.5%
    Failure 2094: m(M)=0.920441, percentile=15.6%
    Failure 2095: m(M)=0.833298, percentile=0.7%
    Failure 2096: m(M)=0.920407, percentile=13.7%
    Failure 2097: m(M)=0.921395, percentile=98.2%
    Failure 2098

    Failure 2338: m(M)=0.920209, percentile=7.7%
    Failure 2339: m(M)=0.658269, percentile=0.2%
    Failure 2340: m(M)=0.917214, percentile=2.7%
    Failure 2341: m(M)=0.917102, percentile=2.7%
    Failure 2342: m(M)=0.861683, percentile=0.9%
    Failure 2343: m(M)=0.915396, percentile=2.2%
    Failure 2344: m(M)=0.920309, percentile=9.8%
    Failure 2345: m(M)=0.606651, percentile=0.1%
    Failure 2346: m(M)=0.833592, percentile=0.7%
    Failure 2347: m(M)=0.882261, percentile=1.1%
    Failure 2348: m(M)=0.867170, percentile=0.9%
    Failure 2349: m(M)=0.771192, percentile=0.4%
    Failure 2350: m(M)=0.919491, percentile=4.5%
    Failure 2351: m(M)=0.910646, percentile=1.7%
    Failure 2352: m(M)=0.906896, percentile=1.6%
    Failure 2353: m(M)=0.904158, percentile=1.5%
    Failure 2354: m(M)=0.920973, percentile=81.2%
    Failure 2355: m(M)=0.912880, percentile=1.9%
    Failure 2356: m(M)=0.916630, percentile=2.5%
    Failure 2357: m(M)=0.905291, percentile=1.5%
    Failure 2358: m

    Failure 2581: m(M)=0.919997, percentile=5.8%
    Failure 2582: m(M)=0.802591, percentile=0.5%
    Failure 2583: m(M)=0.756863, percentile=0.4%
    Failure 2584: m(M)=0.920036, percentile=6.0%
    Failure 2585: m(M)=0.920587, percentile=28.5%
    Failure 2586: m(M)=0.907005, percentile=1.6%
    Failure 2587: m(M)=0.918297, percentile=3.3%
    Failure 2588: m(M)=0.891558, percentile=1.2%
    Failure 2589: m(M)=0.913051, percentile=1.9%
    Failure 2590: m(M)=0.920463, percentile=17.0%
    Failure 2591: m(M)=0.885663, percentile=1.1%
    Failure 2592: m(M)=0.890861, percentile=1.2%
    Failure 2593: m(M)=0.757976, percentile=0.4%
    Failure 2594: m(M)=0.811008, percentile=0.6%
    Failure 2595: m(M)=0.744559, percentile=0.3%
    Failure 2596: m(M)=0.918389, percentile=3.3%
    Failure 2597: m(M)=0.920486, percentile=18.7%
    Failure 2598: m(M)=0.918551, percentile=3.5%
    Failure 2599: m(M)=0.917777, percentile=3.0%
    Failure 2600: m(M)=0.825015, percentile=0.6%
    Failure 2601:

    Failure 2835: m(M)=0.919620, percentile=4.7%
    Failure 2836: m(M)=0.920560, percentile=25.6%
    Failure 2837: m(M)=0.920924, percentile=75.7%
    Failure 2838: m(M)=0.763822, percentile=0.4%
    Failure 2839: m(M)=0.920504, percentile=20.2%
    Failure 2840: m(M)=0.907280, percentile=1.6%
    Failure 2841: m(M)=0.866742, percentile=0.9%
    Failure 2842: m(M)=0.881959, percentile=1.1%
    Failure 2843: m(M)=0.910983, percentile=1.8%
    Failure 2844: m(M)=0.911973, percentile=1.8%
    Failure 2845: m(M)=0.919049, percentile=3.9%
    Failure 2846: m(M)=0.799053, percentile=0.5%
    Failure 2847: m(M)=0.900015, percentile=1.4%
    Failure 2848: m(M)=0.920607, percentile=30.9%
    Failure 2849: m(M)=0.861185, percentile=0.9%
    Failure 2850: m(M)=0.890831, percentile=1.2%
    Failure 2851: m(M)=0.906515, percentile=1.6%
    Failure 2852: m(M)=0.920585, percentile=28.3%
    Failure 2853: m(M)=0.920863, percentile=67.6%
    Failure 2854: m(M)=0.813180, percentile=0.6%
    Failure 28

    Failure 3062: m(M)=0.920035, percentile=6.0%
    Failure 3063: m(M)=0.914081, percentile=2.0%
    Failure 3064: m(M)=0.920787, percentile=56.6%
    Failure 3065: m(M)=0.675381, percentile=0.2%
    Failure 3066: m(M)=0.830533, percentile=0.7%
    Failure 3067: m(M)=0.854826, percentile=0.8%
    Failure 3068: m(M)=0.875044, percentile=1.0%
    Failure 3069: m(M)=0.920607, percentile=30.9%
    Failure 3070: m(M)=0.583421, percentile=0.0%
    Failure 3071: m(M)=0.913637, percentile=2.0%
    Failure 3072: m(M)=0.919501, percentile=4.5%
    Failure 3073: m(M)=0.920901, percentile=72.7%
    Failure 3074: m(M)=0.913590, percentile=2.0%
    Failure 3075: m(M)=0.914194, percentile=2.1%
    Failure 3076: m(M)=0.913115, percentile=1.9%
    Failure 3077: m(M)=0.882541, percentile=1.1%
    Failure 3078: m(M)=0.906627, percentile=1.6%
    Failure 3079: m(M)=0.919121, percentile=4.0%
    Failure 3080: m(M)=0.646311, percentile=0.1%
    Failure 3081: m(M)=0.838772, percentile=0.7%
    Failure 3082:

In [17]:
# =============================================================================
# STAGE 2c: AUTO-SELECT CPL STRATEGY (no labels needed)
# =============================================================================
# For an UNKNOWN dataset without ground truth, we use the spread of m(M)
# scores to decide whether CPL will help and how aggressively to apply it.
#
# Metric: Coefficient of Variation (CV) = std(m(M)) / mean(m(M)) * 100%
#
# Calibrated from benchmark results:
#   BinnedTestIllumina (CV=0.12%): p0 optimal
#   Erlich   (CV=0.01%): p0 optimal
#   Grass    (CV=1.03%): p5 optimal
#   Nanopore 2FC (CV=1.94%): p5/p10 optimal (within noise)
#   Srinivasavaradhan (CV=4.66%): p15 optimal
#
# Adaptive rule (maps CV to percentile):
#   CV < 0.5%          → p0  (skip CPL: DNN errors are confident mistakes)
#   0.5% ≤ CV < 2.0%   → p5  (mild CPL: moderate noise)
#   2.0% ≤ CV < 4.0%   → p10 (moderate CPL: clear low-confidence tail)
#   CV ≥ 4.0%          → p15 (aggressive CPL: high noise, many errors)

if RUN_MODE != "train_only":
    print(f"\n{'='*70}")
    print(f"STAGE 2c: Auto-Select CPL Strategy")
    print(f"{'='*70}\n")

    m_mean = m_arr.mean()
    m_std = m_arr.std()
    m_cv = (m_std / m_mean) * 100  # CV in percent
    m_range = m_arr.max() - m_arr.min()
    m_iqr = np.percentile(m_arr, 75) - np.percentile(m_arr, 25)

    print(f"  Confidence distribution metrics (no labels needed):")
    print(f"    mean(m(M))  = {m_mean:.6f}")
    print(f"    std(m(M))   = {m_std:.6f}")
    print(f"    CV          = {m_cv:.4f}%")
    print(f"    range       = {m_range:.6f}")
    print(f"    IQR         = {m_iqr:.6f}")

    # Adaptive CV-to-percentile mapping
    if m_cv < 0.5:
        AUTO_CPL_PERCENTILE = 0
        cv_reason = "CV < 0.5%: scores tightly clustered, filter cannot separate errors"
    elif m_cv < 2.0:
        AUTO_CPL_PERCENTILE = 5
        cv_reason = "0.5% <= CV < 2.0%: moderate spread, mild CPL recommended"
    elif m_cv < 4.0:
        AUTO_CPL_PERCENTILE = 10
        cv_reason = "2.0% <= CV < 4.0%: clear low-confidence tail, moderate CPL"
    else:
        AUTO_CPL_PERCENTILE = 15
        cv_reason = "CV >= 4.0%: high spread, aggressive CPL recommended"

    print(f"\n  AUTO-SELECT: CV={m_cv:.4f}% -> p{AUTO_CPL_PERCENTILE}")
    print(f"     {cv_reason}")

    if AUTO_CPL_PERCENTILE == 0:
        print(f"     Sending clusters to CPL would likely break correct predictions.")

    print(f"\n  Note: The full sweep (p0..p25) will still run for analysis.")
    print(f"  AUTO_CPL_PERCENTILE={AUTO_CPL_PERCENTILE} is the recommended choice")
    print(f"  for deployment on unseen data from this sequencing technology.")

    # Save the auto-select decision
    auto_select_data = {
        'dataset_name': DATASET_NAME,
        'm_mean': m_mean, 'm_std': m_std, 'm_cv_pct': m_cv,
        'm_range': m_range, 'm_iqr': m_iqr,
        'auto_cpl_percentile': AUTO_CPL_PERCENTILE,
    }
    pd.DataFrame([auto_select_data]).to_csv(
        RESULTS_DIR / f'auto_select_{DATASET_NAME}.csv', index=False)

    sio.savemat(str(MAT_DIR / f'auto_select_{DATASET_NAME}.mat'), {
        'dataset_name': DATASET_NAME,
        'm_mean': np.float64(m_mean),
        'm_std': np.float64(m_std),
        'm_cv_pct': np.float64(m_cv),
        'm_range': np.float64(m_range),
        'm_iqr': np.float64(m_iqr),
        'auto_cpl_percentile': np.float64(AUTO_CPL_PERCENTILE),
    })


STAGE 2c: Auto-Select CPL Strategy

  Confidence distribution metrics (no labels needed):
    mean(m(M))  = 0.918972
    std(m(M))   = 0.017668
    CV          = 1.9225%
    range       = 0.448419
    IQR         = 0.000364

  AUTO-SELECT: CV=1.9225% -> p5
     0.5% <= CV < 2.0%: moderate spread, mild CPL recommended

  Note: The full sweep (p0..p25) will still run for analysis.
  AUTO_CPL_PERCENTILE=5 is the recommended choice
  for deployment on unseen data from this sequencing technology.


In [18]:
# =============================================================================
# STAGE 3: Multi-Threshold CPL Sweep
# =============================================================================

if RUN_MODE != "train_only":
    print(f"\n{'='*70}")
    print(f"STAGE 3: CPL Sweep (clustersizethreshold={CLUSTERSIZE_THRESHOLD})")
    print(f"{'='*70}\n")
    max_pct = max(CPL_PERCENTILES)
    if max_pct == 0 or dnn_failed == 0:
        print("  Skipping CPL"); cpl_predictions = {}
    else:
        max_threshold = np.percentile(m_arr, max_pct)
        cf_max = ConfidenceFilter(threshold=max_threshold, min_cluster_for_cpl=CLUSTERSIZE_THRESHOLD)
        all_cpl_idx = []; all_era_idx = []
        for i in range(N):
            d, _ = cf_max.classify(all_probs[i], test_ds.get_cluster_size(i))
            if d == 'send_to_cpl': all_cpl_idx.append(i)
            elif d == 'erasure': all_era_idx.append(i)
        print(f'  Superset at p{max_pct}: Trust={N-len(all_cpl_idx)-len(all_era_idx):,} CPL={len(all_cpl_idx):,} Era={len(all_era_idx):,}')

        cpl_predictions = {}; cpl_fx = 0; cpl_bk = 0; cpl_er = 0; t0 = time.time()
        for ii, i in enumerate(tqdm(all_cpl_idx, desc="CPL")):
            reads = test_ds.get_raw_reads(i)
            try: cp = run_cpl_on_cluster(reads, ORIGINAL_SEQ_LEN, output_length=LABEL_SEQ_LEN)
            except: cp = dnn_decoded[i]; cpl_er += 1
            cpl_predictions[i] = cp
            if dnn_decoded[i] != label_decoded[i] and cp == label_decoded[i]: cpl_fx += 1
            elif dnn_decoded[i] == label_decoded[i] and cp != label_decoded[i]: cpl_bk += 1
            if (ii+1) % 1000 == 0:
                el = time.time()-t0; rate = (ii+1)/el; rem = (len(all_cpl_idx)-ii-1)/rate
                print(f'  [{ii+1}/{len(all_cpl_idx)}] {rate:.1f} cl/s | fixed={cpl_fx} broke={cpl_bk} | ETA: {rem/60:.0f}min')
        print(f"\n  Done: {time.time()-t0:.0f}s | fixed={cpl_fx} broke={cpl_bk} net={'+' if cpl_fx>=cpl_bk else ''}{cpl_fx-cpl_bk}")



STAGE 3: CPL Sweep (clustersizethreshold=2)

  Superset at p25: Trust=82,491 CPL=27,442 Era=43


CPL:   0%|          | 0/27442 [00:00<?, ?it/s]

  [1000/27442] 0.3 cl/s | fixed=65 broke=18 | ETA: 1490min
  [2000/27442] 0.3 cl/s | fixed=120 broke=33 | ETA: 1416min
  [3000/27442] 0.3 cl/s | fixed=178 broke=50 | ETA: 1343min
  [4000/27442] 0.3 cl/s | fixed=245 broke=70 | ETA: 1285min
  [5000/27442] 0.3 cl/s | fixed=294 broke=79 | ETA: 1231min
  [6000/27442] 0.3 cl/s | fixed=357 broke=93 | ETA: 1172min
  [7000/27442] 0.3 cl/s | fixed=419 broke=111 | ETA: 1113min
  [8000/27442] 0.3 cl/s | fixed=489 broke=128 | ETA: 1057min
  [9000/27442] 0.3 cl/s | fixed=537 broke=139 | ETA: 1002min
  [10000/27442] 0.3 cl/s | fixed=600 broke=149 | ETA: 947min
  [11000/27442] 0.3 cl/s | fixed=687 broke=168 | ETA: 887min
  [12000/27442] 0.3 cl/s | fixed=769 broke=185 | ETA: 828min
  [13000/27442] 0.3 cl/s | fixed=820 broke=215 | ETA: 777min
  [14000/27442] 0.3 cl/s | fixed=888 broke=236 | ETA: 723min
  [15000/27442] 0.3 cl/s | fixed=968 broke=250 | ETA: 671min
  [16000/27442] 0.3 cl/s | fixed=1023 broke=261 | ETA: 618min
  [17000/27442] 0.3 cl/s | fix

In [19]:
# =============================================================================
# STAGE 4: Per-Threshold Results
# =============================================================================

if RUN_MODE != "train_only":
    print(f"\n{'='*70}")
    print(f"STAGE 4: Per-Threshold Results")
    print(f"{'='*70}\n")
    threshold_results = []
    for pct in CPL_PERCENTILES:
        if pct == 0:
            ff = dnn_failed; fr = dnn_rate; nt = N; nc = 0; ne = 0; cf_ = 0; cb_ = 0
        else:
            thr = np.percentile(m_arr, pct)
            cf = ConfidenceFilter(threshold=thr, min_cluster_for_cpl=CLUSTERSIZE_THRESHOLD)
            tp, cp, ep = [], [], []
            for i in range(N):
                d, _ = cf.classify(all_probs[i], test_ds.get_cluster_size(i))
                if d == 'trust_dnn': tp.append(i)
                elif d == 'send_to_cpl': cp.append(i)
                else: ep.append(i)
            nt = len(tp); nc = len(cp); ne = len(ep)
            es = set(ep); cs_set = set(cp); ff = 0; cf_ = 0; cb_ = 0
            for i in range(N):
                lb = label_decoded[i]
                if i in es: ff += 1
                elif i in cs_set and i in cpl_predictions:
                    cpred = cpl_predictions[i]
                    if cpred != lb: ff += 1
                    if dnn_decoded[i] != lb and cpred == lb: cf_ += 1
                    elif dnn_decoded[i] == lb and cpred != lb: cb_ += 1
                else:
                    if dnn_decoded[i] != lb: ff += 1
            fr = ff / N * 100
        imp = dnn_failed - ff
        r = {'percentile':pct, 'threshold':np.percentile(m_arr,pct) if pct>0 else None,
             'total':N, 'dnn_failed':dnn_failed, 'dnn_rate_pct':dnn_rate,
             'final_failed':ff, 'final_rate_pct':fr, 'improvement':imp,
             'trust_dnn':nt, 'sent_to_cpl':nc, 'erasures':ne,
             'cpl_fixed':cf_, 'cpl_broke':cb_, 'cpl_net':cf_-cb_,
             'target_pct':CONFIG['target_failure'], 'target_achieved':fr<=CONFIG['target_failure'],
             'dataset':DATASET_NAME, 'clustersize_threshold':CLUSTERSIZE_THRESHOLD}
        threshold_results.append(r)
        sign = '+' if imp > 0 else ''
        st = '🎉 TARGET!' if r['target_achieved'] else f"gap:{fr-CONFIG['target_failure']:.4f}%"
        print(f"  p{pct:>2}: failed={ff:>6,} ({fr:.4f}%) | fix={cf_} brk={cb_} net={sign}{cf_-cb_} | sent={nc:,} era={ne} | {st}")

    results_df = pd.DataFrame(threshold_results)
    results_df.to_csv(RESULTS_DIR / f'results_multithreshold_{DATASET_NAME}.csv', index=False)

    sio.savemat(str(MAT_DIR / f'fig5_sweep_{DATASET_NAME}.mat'), {
        'dataset_name': DATASET_NAME,
        'percentiles': np.array([r['percentile'] for r in threshold_results], dtype=np.float64),
        'final_failed': np.array([r['final_failed'] for r in threshold_results], dtype=np.float64),
        'final_rate_pct': np.array([r['final_rate_pct'] for r in threshold_results], dtype=np.float64),
        'cpl_fixed': np.array([r['cpl_fixed'] for r in threshold_results], dtype=np.float64),
        'cpl_broke': np.array([r['cpl_broke'] for r in threshold_results], dtype=np.float64),
        'cpl_net': np.array([r['cpl_net'] for r in threshold_results], dtype=np.float64),
        'dnn_rate_pct': np.float64(dnn_rate), 'dnn_failed': np.float64(dnn_failed),
        'target_failure_pct': np.float64(CONFIG['target_failure']),
        'clustersize_threshold': np.float64(CLUSTERSIZE_THRESHOLD), 'N': np.float64(N),
    })
    print(f"\n  → Saved results + .mat")



STAGE 4: Per-Threshold Results

  p 0: failed= 3,139 (2.8543%) | fix=0 brk=0 net=0 | sent=0 era=0 | gap:1.2043%
  p 5: failed= 1,788 (1.6258%) | fix=1590 brk=236 net=+1354 | sent=5,456 era=43 | 🎉 TARGET!
  p10: failed= 1,809 (1.6449%) | fix=1686 brk=353 net=+1333 | sent=10,953 era=43 | 🎉 TARGET!
  p15: failed= 1,818 (1.6531%) | fix=1717 brk=393 net=+1324 | sent=16,454 era=43 | gap:0.0031%
  p20: failed= 1,825 (1.6595%) | fix=1736 brk=419 net=+1317 | sent=21,946 era=43 | gap:0.0095%
  p25: failed= 1,837 (1.6704%) | fix=1748 brk=443 net=+1305 | sent=27,442 era=43 | gap:0.0204%

  → Saved results + .mat


In [20]:
# =============================================================================
# FINAL RESULTS + RECOMMENDATION
# =============================================================================

if RUN_MODE != "train_only":
    best = min(threshold_results, key=lambda r: r['final_failed'])
    best_pct = best['percentile']
    print(f"\n{'='*70}")
    print(f"FINAL RESULTS: {DATASET_NAME}  (clustersizethreshold={CLUSTERSIZE_THRESHOLD})")
    print(f"{'='*70}")
    print(f"\n  Total: {N:,} | DNN-only: {dnn_failed:,} ({dnn_rate:.4f}%)")
    print(f"\n  {'Pctl':>5} {'Failed':>8} {'Rate%':>10} {'Net':>6} {'Status':>12}")
    print(f"  {'-'*5} {'-'*8} {'-'*10} {'-'*6} {'-'*12}")
    for r in threshold_results:
        sign = '+' if r['improvement'] > 0 else ''
        st = '🎉 TARGET' if r['target_achieved'] else ''
        pl = f"p{r['percentile']}" if r['percentile'] > 0 else 'DNN'
        print(f"  {pl:>5} {r['final_failed']:>8,} {r['final_rate_pct']:>10.4f} {sign}{r['improvement']:>5} {st:>12}")
    print(f"\n  Best: p{best_pct} → {best['final_failed']} ({best['final_rate_pct']:.4f}%) | Target: {CONFIG['target_failure']}%")
    if best['target_achieved']: print('  🎉 TARGET ACHIEVED!')
    else: print(f"  📈 Gap: {best['final_rate_pct']-CONFIG['target_failure']:.4f}%")
    print(f"{'='*70}")

    # Recommendation (including auto-select for unknown datasets)
    print(f"\n  AUTO-SELECT (for unknown datasets, no labels needed):")
    print(f"    CV of m(M) = {m_cv:.4f}%")
    print(f"    → p{AUTO_CPL_PERCENTILE} ({'skip CPL' if AUTO_CPL_PERCENTILE == 0 else 'use CPL'})")
    print()
    print(f"  ORACLE (with labels, from sweep):")
    print(f"\n  RECOMMENDATION:")
    print(f"  clustersizethreshold = {CLUSTERSIZE_THRESHOLD}")
    dnn_only_r = [r for r in threshold_results if r['percentile'] == 0][0]
    if dnn_failed == 0:
        print("  → CPL_PERCENTILE = 0 (zero DNN failures)")
    else:
        helps = any(r['final_failed'] < dnn_only_r['final_failed'] for r in threshold_results if r['percentile'] > 0)
        if not helps:
            print(f"  → CPL_PERCENTILE = 0 (no threshold improves over DNN-only)")
        else:
            br = min(threshold_results, key=lambda r: r['final_failed'])
            print(f"  → CPL_PERCENTILE = {br['percentile']} (best: {br['final_failed']} failures, {br['final_rate_pct']:.4f}%)")

    # Save combined .mat
    sio.savemat(str(MAT_DIR / f'combined_{DATASET_NAME}.mat'), {
        'dataset_name': DATASET_NAME, 'N': np.float64(N),
        'dnn_failed': np.float64(dnn_failed), 'dnn_rate_pct': np.float64(dnn_rate),
        'target_failure_pct': np.float64(CONFIG['target_failure']),
        'clustersize_threshold': np.float64(CLUSTERSIZE_THRESHOLD),
        'cs_sizes': np.array(all_sizes_sorted, dtype=np.float64),
        'cs_success_rates': np.array([size_success_rate[cs] for cs in all_sizes_sorted], dtype=np.float64),
        'm_scores_all': m_arr, 'm_scores_correct': correct_scores, 'm_scores_failed': fail_scores,
        'cluster_sizes_all': cluster_sizes.astype(np.float64),
        'dnn_correct_mask': dnn_correct_mask.astype(np.float64),
        'sweep_percentiles': np.array([r['percentile'] for r in threshold_results], dtype=np.float64),
        'sweep_final_rate': np.array([r['final_rate_pct'] for r in threshold_results], dtype=np.float64),
        'sweep_cpl_fixed': np.array([r['cpl_fixed'] for r in threshold_results], dtype=np.float64),
        'sweep_cpl_broke': np.array([r['cpl_broke'] for r in threshold_results], dtype=np.float64),
        'sweep_cpl_net': np.array([r['cpl_net'] for r in threshold_results], dtype=np.float64),
        'best_percentile': np.float64(best_pct),
        'best_final_rate': np.float64(best['final_rate_pct']),
    })
    print(f"\n  → Saved combined .mat: {MAT_DIR}/combined_{DATASET_NAME}.mat")



FINAL RESULTS: BinnedNanoporeTwoFlowcells  (clustersizethreshold=2)

  Total: 109,976 | DNN-only: 3,139 (2.8543%)

   Pctl   Failed      Rate%    Net       Status
  ----- -------- ---------- ------ ------------
    DNN    3,139     2.8543     0             
     p5    1,788     1.6258 + 1351     🎉 TARGET
    p10    1,809     1.6449 + 1330     🎉 TARGET
    p15    1,818     1.6531 + 1321             
    p20    1,825     1.6595 + 1314             
    p25    1,837     1.6704 + 1302             

  Best: p5 → 1788 (1.6258%) | Target: 1.65%
  🎉 TARGET ACHIEVED!

  AUTO-SELECT (for unknown datasets, no labels needed):
    CV of m(M) = 1.9225%
    → p5 (use CPL)

  ORACLE (with labels, from sweep):

  RECOMMENDATION:
  clustersizethreshold = 2
  → CPL_PERCENTILE = 5 (best: 1788 failures, 1.6258%)

  → Saved combined .mat: Experiments/BinnedNanoporeTwoFlowcells_v4/MatFiles/combined_BinnedNanoporeTwoFlowcells.mat


In [21]:
# =============================================================================
# APPENDIX: DNAformer-Style Confidence Filter (exact parameters from paper)
# =============================================================================
# For comparison: apply DNAformer's EXACT confidence filter formulation
#   c(M, K) = m(M)^{2*K}
# with their published per-technology thresholds:
#   Illumina:  confidencethreshold = 0,   clustersizethreshold = 3
#   Nanopore:  confidencethreshold = 0.7,  clustersizethreshold = 4
#
# For public benchmarks (Erlich, Grass), DNAformer did not specify parameters.
# We apply Illumina params for Illumina-sequenced datasets and
# Nanopore params for Nanopore-sequenced datasets.

DNAFORMER_PARAMS = {
    "BinnedTestIllumina":          {"conf_thr": 0.0, "cs_thr": 3, "tech": "Illumina"},
    "BinnedPilotIllumina":         {"conf_thr": 0.0, "cs_thr": 3, "tech": "Illumina"},
    "BinnedNanoporeTwoFlowcells":  {"conf_thr": 0.7, "cs_thr": 4, "tech": "Nanopore"},
    "BinnedNanoporeFirstFlowcell": {"conf_thr": 0.7, "cs_thr": 4, "tech": "Nanopore"},
    "BinnedNanoporeSecondFlowcell":{"conf_thr": 0.7, "cs_thr": 4, "tech": "Nanopore"},
    "BinnedPilotNanopore":         {"conf_thr": 0.7, "cs_thr": 4, "tech": "Nanopore"},
    "Erlich":                      {"conf_thr": 0.0, "cs_thr": 3, "tech": "Illumina"},
    "Grass":                       {"conf_thr": 0.0, "cs_thr": 3, "tech": "Illumina"},
    "Srinivasavaradhan":           {"conf_thr": 0.7, "cs_thr": 4, "tech": "Nanopore"},
}

if RUN_MODE != "train_only" and DATASET_NAME in DNAFORMER_PARAMS:
    print(f"\n{'='*70}")
    print(f"APPENDIX: DNAformer-Style Confidence Filter Comparison")
    print(f"{'='*70}\n")

    df_p = DNAFORMER_PARAMS[DATASET_NAME]
    df_conf_thr = df_p['conf_thr']
    df_cs_thr = df_p['cs_thr']
    df_tech = df_p['tech']

    print(f"  DNAformer parameters for {DATASET_NAME} ({df_tech}):")
    print(f"    confidencethreshold  = {df_conf_thr}")
    print(f"    clustersizethreshold = {df_cs_thr}")
    print(f"    Formula: c(M,K) = m(M)^(2*K)")

    # Compute DNAformer's c(M,K) for each cluster
    df_trust = []; df_cpl = []; df_erasure = []
    for i in range(N):
        m_M = m_scores[i]
        K_i = cluster_sizes[i]
        c_val = m_M ** (2 * K_i)
        if c_val > df_conf_thr:
            df_trust.append(i)
        elif K_i <= df_cs_thr:
            df_erasure.append(i)
        else:
            df_cpl.append(i)

    print(f"\n  DNAformer filter classification:")
    print(f"    Trust DNN: {len(df_trust):,}")
    print(f"    Send CPL:  {len(df_cpl):,}")
    print(f"    Erasures:  {len(df_erasure):,}")

    if df_conf_thr == 0.0:
        print(f"\n  With confidencethreshold=0, c(M,K) > 0 always → nothing sent to CPL.")
        print(f"  DNAformer result = DNN-only: {dnn_failed} failures ({dnn_rate:.4f}%)")
        df_final_failed = dnn_failed
        df_final_rate = dnn_rate
        df_cpl_fixed = 0; df_cpl_broke = 0
    else:
        # Run CPL on DNAformer's selected clusters
        df_cpl_preds = {}
        df_cpl_set = set(df_cpl)
        reused = 0; need_new = 0

        for i in df_cpl:
            if i in cpl_predictions:
                df_cpl_preds[i] = cpl_predictions[i]
                reused += 1
            else:
                need_new += 1

        print(f"\n  CPL: {reused} reused from our sweep, {need_new} need new computation")

        if need_new > 0:
            print(f"  Running CPL on {need_new} additional clusters...")
            new_list = [i for i in df_cpl if i not in cpl_predictions]
            for i in tqdm(new_list, desc='DNAformer CPL'):
                reads = test_ds.get_raw_reads(i)
                try:
                    cp = run_cpl_on_cluster(reads, ORIGINAL_SEQ_LEN, output_length=LABEL_SEQ_LEN)
                except:
                    cp = dnn_decoded[i]
                df_cpl_preds[i] = cp

        # Compute final results with DNAformer's filter
        df_era_set = set(df_erasure)
        df_final_failed = 0; df_cpl_fixed = 0; df_cpl_broke = 0
        for i in range(N):
            lb = label_decoded[i]
            if i in df_era_set:
                df_final_failed += 1
            elif i in df_cpl_set and i in df_cpl_preds:
                cpred = df_cpl_preds[i]
                if cpred != lb: df_final_failed += 1
                if dnn_decoded[i] != lb and cpred == lb: df_cpl_fixed += 1
                elif dnn_decoded[i] == lb and cpred != lb: df_cpl_broke += 1
            else:
                if dnn_decoded[i] != lb: df_final_failed += 1
        df_final_rate = df_final_failed / N * 100

        print(f"\n  DNAformer CPL results:")
        print(f"    Fixed: {df_cpl_fixed}, Broke: {df_cpl_broke}, Net: {'+' if df_cpl_fixed>=df_cpl_broke else ''}{df_cpl_fixed-df_cpl_broke}")

    # Find our results for comparison
    our_best = min(threshold_results, key=lambda r: r['final_failed'])
    our_auto = [r for r in threshold_results if r['percentile'] == AUTO_CPL_PERCENTILE][0]

    # Print comparison table
    print(f"\n  {'='*65}")
    print(f"  COMPARISON: DNAformer-style vs Our Adaptive Approach")
    print(f"  {'='*65}")
    print(f"  {'Method':<35} {'Failed':>8} {'Rate%':>10} {'Impr':>8}")
    print(f"  {'-'*35} {'-'*8} {'-'*10} {'-'*8}")
    print(f"  {'DNN-only (baseline)':<35} {dnn_failed:>8,} {dnn_rate:>10.4f} {'---':>8}")
    df_imp = dnn_failed - df_final_failed
    df_sign = '+' if df_imp > 0 else ''
    print(f"  {'DNAformer (c=m^2K, thr='+str(df_conf_thr)+')':<35} {df_final_failed:>8,} {df_final_rate:>10.4f} {df_sign+str(df_imp):>8}")
    au_sign = '+' if our_auto['improvement'] > 0 else ''
    print(f"  {'Ours CV-auto (p'+str(AUTO_CPL_PERCENTILE)+')':<35} {our_auto['final_failed']:>8,} {our_auto['final_rate_pct']:>10.4f} {au_sign+str(our_auto['improvement']):>8}")
    or_sign = '+' if our_best['improvement'] > 0 else ''
    print(f"  {'Ours oracle (p'+str(our_best['percentile'])+')':<35} {our_best['final_failed']:>8,} {our_best['final_rate_pct']:>10.4f} {or_sign+str(our_best['improvement']):>8}")
    print(f"  {'Target':<35} {'---':>8} {CONFIG['target_failure']:>10.4f} {'---':>8}")
    print(f"  {'='*65}")

    # Save
    comp = {
        'dataset': DATASET_NAME, 'technology': df_tech, 'N': N,
        'dnn_failed': dnn_failed, 'dnn_rate': dnn_rate,
        'dnaformer_conf_thr': df_conf_thr, 'dnaformer_cs_thr': df_cs_thr,
        'dnaformer_sent_cpl': len(df_cpl), 'dnaformer_erasures': len(df_erasure),
        'dnaformer_failed': df_final_failed, 'dnaformer_rate': df_final_rate,
        'dnaformer_fixed': df_cpl_fixed, 'dnaformer_broke': df_cpl_broke,
        'ours_auto_p': AUTO_CPL_PERCENTILE, 'ours_auto_cv': m_cv,
        'ours_auto_failed': our_auto['final_failed'], 'ours_auto_rate': our_auto['final_rate_pct'],
        'ours_oracle_p': our_best['percentile'],
        'ours_oracle_failed': our_best['final_failed'], 'ours_oracle_rate': our_best['final_rate_pct'],
        'target': CONFIG['target_failure'],
    }
    pd.DataFrame([comp]).to_csv(RESULTS_DIR / f'comparison_dnaformer_vs_ours_{DATASET_NAME}.csv', index=False)

    sio.savemat(str(MAT_DIR / f'comparison_{DATASET_NAME}.mat'), {
        'dataset_name': DATASET_NAME,
        'dnn_failed': np.float64(dnn_failed), 'dnn_rate': np.float64(dnn_rate),
        'dnaformer_conf_thr': np.float64(df_conf_thr), 'dnaformer_cs_thr': np.float64(df_cs_thr),
        'dnaformer_sent_cpl': np.float64(len(df_cpl)), 'dnaformer_erasures': np.float64(len(df_erasure)),
        'dnaformer_failed': np.float64(df_final_failed), 'dnaformer_rate': np.float64(df_final_rate),
        'dnaformer_fixed': np.float64(df_cpl_fixed), 'dnaformer_broke': np.float64(df_cpl_broke),
        'ours_auto_p': np.float64(AUTO_CPL_PERCENTILE), 'ours_auto_cv': np.float64(m_cv),
        'ours_auto_failed': np.float64(our_auto['final_failed']),
        'ours_auto_rate': np.float64(our_auto['final_rate_pct']),
        'ours_oracle_p': np.float64(our_best['percentile']),
        'ours_oracle_failed': np.float64(our_best['final_failed']),
        'ours_oracle_rate': np.float64(our_best['final_rate_pct']),
        'target': np.float64(CONFIG['target_failure']),
    })
    print(f"\n  → Saved comparison CSV + .mat")

elif RUN_MODE != "train_only":
    print(f"\n  {DATASET_NAME} not in DNAformer parameter table, skipping comparison.")




APPENDIX: DNAformer-Style Confidence Filter Comparison

  DNAformer parameters for BinnedNanoporeTwoFlowcells (Nanopore):
    confidencethreshold  = 0.7
    clustersizethreshold = 4
    Formula: c(M,K) = m(M)^(2*K)

  DNAformer filter classification:
    Trust DNN: 1
    Send CPL:  109,761
    Erasures:  214

  CPL: 27314 reused from our sweep, 82447 need new computation
  Running CPL on 82447 additional clusters...


DNAformer CPL:   0%|          | 0/82447 [00:00<?, ?it/s]


  DNAformer CPL results:
    Fixed: 1837, Broke: 673, Net: +1164

  COMPARISON: DNAformer-style vs Our Adaptive Approach
  Method                                Failed      Rate%     Impr
  ----------------------------------- -------- ---------- --------
  DNN-only (baseline)                    3,139     2.8543      ---
  DNAformer (c=m^2K, thr=0.7)            2,082     1.8931    +1057
  Ours CV-auto (p5)                      1,788     1.6258    +1351
  Ours oracle (p5)                       1,788     1.6258    +1351
  Target                                   ---     1.6500      ---

  → Saved comparison CSV + .mat
